# LangGraph + RAG 智能助手教程

本教程演示如何使用 LangGraph 构建一个具备多种能力的智能助手，包括：
- 🌤️ 天气查询
- 🧮 数学计算
- 📚 文档检索（RAG）

## 架构说明

这个应用使用 **LangGraph** 来编排工作流，通过条件路由将不同类型的工具调用分发到相应的节点处理。


## 1. 导入依赖库

首先导入所有需要的库和模块。


In [3]:
from langchain_community.document_loaders import WebBaseLoader
from langgraph.prebuilt.interrupt import HumanInterruptConfig
from langgraph.func import entrypoint, task
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langgraph.prebuilt import ToolNode
from langchain.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_core.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
import operator


## 2. 构建向量数据库（RAG 知识库）

这一步我们从网页加载文档，将其分块并向量化，创建一个可以检索的知识库。

**步骤说明：**
1. 使用 `WebBaseLoader` 加载联想官网内容
2. 使用 `RecursiveCharacterTextSplitter` 将文档切分成小块
3. 使用 OpenAI Embeddings 将文本向量化
4. 使用 FAISS 创建向量数据库
5. 创建检索器（retriever）


In [ ]:
# 加载文档并分块
loader = WebBaseLoader("https://www.lenovo.com.cn/")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = text_splitter.split_documents(documents)

# 向量化并创建向量库
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small", 
    api_key="api_key"
)
vector_store = FAISS.from_documents(splits, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

print(f"✅ 文档加载完成！共分割成 {len(splits)} 个文本块")


✅ 文档加载完成！共分割成 71 个文本块


## 3. 定义状态结构

LangGraph 使用状态（State）来在节点之间传递信息。这里我们定义了包含消息历史和 LLM 调用次数的状态。


In [5]:
class State(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]  # 对话历史
    llm_calls: int  # LLM 调用次数计数


## 4. 定义工具函数

我们定义三个工具函数供 LLM 调用：
1. **weather_tool** - 查询天气信息
2. **calc_tool** - 执行数学计算
3. **retrieve_docs** - 从向量数据库检索文档

使用 `@tool` 装饰器可以让 LangChain 自动将函数转换为可调用的工具。


In [6]:
@tool
def weather_tool(city: str) -> str:
    """查询天气：返回指定城市天气信息。"""
    # 这里使用模拟数据，实际可调用天气API
    return f"{city}：多云，20°C"

@tool
def calc_tool(x: float, y: float, op: str) -> float:
    """简单计算器：加减乘除运算。"""
    if op == "+": return x + y
    if op == "-": return x - y
    if op == "*": return x * y
    if op == "/": return x / y
    raise ValueError("运算符无效")

@tool
def retrieve_docs(query: str) -> str:
    """检索企业文档：返回与 query 相关的文档内容。"""
    docs = retriever.invoke(query)
    return "\n".join([doc.page_content for doc in docs])

# 分离工具列表
general_tools = [weather_tool, calc_tool]  # 一般工具
retrieval_tools = [retrieve_docs]  # 检索工具
all_tools = general_tools + retrieval_tools  # 所有工具（用于模型绑定）

print(f"✅ 工具定义完成！共 {len(all_tools)} 个工具")


✅ 工具定义完成！共 3 个工具


## 5. 初始化记忆组件

为了让助手能够记住对话历史，我们使用：
- **InMemorySaver** - 保存检查点（checkpoint）
- **InMemoryStore** - 存储额外数据


In [7]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

checkpointer = InMemorySaver()
store = InMemoryStore()


## 6. 创建 LangGraph 图结构

这是核心部分！我们创建一个状态图（StateGraph），定义：
- **节点（Node）** - 执行具体任务的单元
- **边（Edge）** - 连接节点的路径
- **条件路由** - 根据条件决定下一步走向


In [8]:
from langgraph.graph import StateGraph, START, END

# 创建图构建器
graph_builder = StateGraph(State)


### 6.1 定义系统提示和 LLM 节点

系统提示告诉 AI 它有哪些能力以及如何使用工具。


In [9]:
SYSTEM_PROMPT = """你是一个智能助手，具备以下能力：
1. 天气查询：使用 weather_tool 查询城市天气，参数为城市名称
2. 数学计算：使用 calc_tool 进行计算，参数为两个数字和运算符
3. 文档检索：使用 retrieve_docs 检索联想官网信息，参数为查询关键词

当用户需要查询天气、进行计算或查找文档信息时，请调用相应的工具。
其他情况下直接回答用户问题。"""

# 创建模型并绑定工具
chat_model = ChatOpenAI(
    model="Qwen3-235B-A22B-Instruct-2507", 
    temperature=0, 
    base_url="http://103.212.13.119:8000/v1", 
    api_key="qwertiasagv"
)

# 关键：将所有工具绑定到模型
chat_model_with_tools = chat_model.bind_tools(all_tools)

def llm_node(state: State) -> dict:
    """调用模型，决策下一步是否使用工具或检索。"""
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = chat_model_with_tools.invoke(messages)
    return {"messages": [response], "llm_calls": state.get("llm_calls", 0) + 1}


### 6.2 添加节点到图中

我们定义四个节点：
1. **agent** - LLM 决策节点
2. **general_tools** - 通用工具节点（天气、计算）
3. **retrieval_tools** - 检索工具节点（文档检索）
4. **final** - 最终输出节点


In [10]:
graph_builder.add_node("agent", llm_node)
graph_builder.add_node("general_tools", ToolNode(general_tools))
graph_builder.add_node("retrieval_tools", ToolNode(retrieval_tools))
graph_builder.add_node("final", lambda st: {"messages": st["messages"]})


### 6.3 定义条件路由函数

这个函数检查 LLM 的输出，判断是否需要调用工具，以及调用哪种类型的工具。


In [11]:
def react_condition(state: State):
    """ReAct 逻辑判断函数：检查模型调用了哪种类型的工具"""
    last_msg = state["messages"][-1]
    
    # 调试信息
    print(f"[DEBUG] 检查消息类型: {type(last_msg)}")
    print(f"[DEBUG] 消息内容: {last_msg.content}")
    
    # 检查消息是否包含工具调用
    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
        tool_call = last_msg.tool_calls[0]
        tool_name = tool_call["name"]
        print(f"[DEBUG] 发现工具调用: {tool_name}")
        
        # 根据工具名称判断路由到哪个节点
        if tool_name == "retrieve_docs":
            return "retrieval_tools"
        elif tool_name in ["weather_tool", "calc_tool"]:
            return "general_tools"
        else:
            print(f"[DEBUG] 未知工具: {tool_name}")
            return "__end__"
    else:
        print("[DEBUG] 没有工具调用，直接结束")
        return "__end__"


### 6.4 设置图的边和流程

定义节点之间的连接关系和流程控制。


In [12]:
# 入口：从 START 进入 agent
graph_builder.add_edge(START, "agent")

# agent 根据条件路由到不同节点
graph_builder.add_conditional_edges(
    "agent",
    react_condition,
    {
        "general_tools": "general_tools",
        "retrieval_tools": "retrieval_tools", 
        "__end__": "final"
    },
)

# 工具执行完后返回 agent
graph_builder.add_edge("general_tools", "agent")
graph_builder.add_edge("retrieval_tools", "agent")

# 最终节点输出
graph_builder.add_edge("final", END)


## 7. 编译图

将图编译成可执行的应用，配置记忆功能。


In [13]:
graph = graph_builder.compile(checkpointer=checkpointer, store=store)

print("✅ LangGraph 图构建完成！")


✅ LangGraph 图构建完成！


## 8. 测试运行

现在我们可以测试我们的智能助手了！试试不同类型的查询：


In [14]:
# 测试函数
def test_query(user_input: str, thread_id: str = "test_1"):
    """测试单个查询"""
    print(f"\n{'='*60}")
    print(f"用户: {user_input}")
    print('='*60)
    
    state = {"messages": [HumanMessage(content=user_input)], "llm_calls": 0}
    config = {"configurable": {"thread_id": thread_id, "user_id": "u123"}}
    
    result = graph.invoke(state, config)["messages"][-1]
    print(f"\n助手: {result.content}")
    return result


### 测试 1：文档检索（RAG）


In [15]:
test_query("联想官网现在卖什么东西？")



用户: 联想官网现在卖什么东西？
[DEBUG] 检查消息类型: <class 'langchain_core.messages.ai.AIMessage'>
[DEBUG] 消息内容: 
[DEBUG] 发现工具调用: retrieve_docs
[DEBUG] 检查消息类型: <class 'langchain_core.messages.ai.AIMessage'>
[DEBUG] 消息内容: 联想官网目前销售的产品主要包括以下几类：

1. **电脑产品**：
   - **Lenovo电脑**：包括笔记本电脑、台式机等。
   - **ThinkPad电脑**：主打商务系列的笔记本电脑。

2. **移动与智能设备**：
   - **平板电脑**
   - **手机/通信设备**

3. **智能家居**：
   - **全屋智能**：提供智能生活解决方案。

4. **配件与办公用品**：
   - 各类电脑配件、外设、办公设备等。

5. **增值服务**：
   - 延长保修、技术支持、企业采购服务（如联想企业购）、教育特惠（如大学生优惠）等。

此外，官网还提供：
- 驱动与软件下载
- 保修查询与服务支持
- 线下门店查询
- 联想社区与技术支持工具（如Lenovo Quick Fix）

您也可以通过 lecoo 商城、联想惠采、联想E采等平台获取更多采购和服务选项。
[DEBUG] 没有工具调用，直接结束

助手: 联想官网目前销售的产品主要包括以下几类：

1. **电脑产品**：
   - **Lenovo电脑**：包括笔记本电脑、台式机等。
   - **ThinkPad电脑**：主打商务系列的笔记本电脑。

2. **移动与智能设备**：
   - **平板电脑**
   - **手机/通信设备**

3. **智能家居**：
   - **全屋智能**：提供智能生活解决方案。

4. **配件与办公用品**：
   - 各类电脑配件、外设、办公设备等。

5. **增值服务**：
   - 延长保修、技术支持、企业采购服务（如联想企业购）、教育特惠（如大学生优惠）等。

此外，官网还提供：
- 驱动与软件下载
- 保修查询与服务支持
- 线下门店查询
- 联想社区与技术支持工具（如Lenovo Quick Fix）

您也可以通

AIMessage(content='联想官网目前销售的产品主要包括以下几类：\n\n1. **电脑产品**：\n   - **Lenovo电脑**：包括笔记本电脑、台式机等。\n   - **ThinkPad电脑**：主打商务系列的笔记本电脑。\n\n2. **移动与智能设备**：\n   - **平板电脑**\n   - **手机/通信设备**\n\n3. **智能家居**：\n   - **全屋智能**：提供智能生活解决方案。\n\n4. **配件与办公用品**：\n   - 各类电脑配件、外设、办公设备等。\n\n5. **增值服务**：\n   - 延长保修、技术支持、企业采购服务（如联想企业购）、教育特惠（如大学生优惠）等。\n\n此外，官网还提供：\n- 驱动与软件下载\n- 保修查询与服务支持\n- 线下门店查询\n- 联想社区与技术支持工具（如Lenovo Quick Fix）\n\n您也可以通过 lecoo 商城、联想惠采、联想E采等平台获取更多采购和服务选项。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 222, 'prompt_tokens': 913, 'total_tokens': 1135, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen3-235B-A22B-Instruct-2507', 'system_fingerprint': None, 'id': 'chatcmpl-7dae34978f5d4370a2163022e321c16c', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--90193b85-e0e2-4946-81bf-ac5adeb3ea8d-0', usage_metadata={'input_tokens': 913, 'output_tokens': 222, 'total_tokens': 1135, 'input_to

### 测试 2：天气查询


In [16]:
test_query("北京今天天气怎么样？")



用户: 北京今天天气怎么样？
[DEBUG] 检查消息类型: <class 'langchain_core.messages.ai.AIMessage'>
[DEBUG] 消息内容: 
[DEBUG] 发现工具调用: weather_tool
[DEBUG] 检查消息类型: <class 'langchain_core.messages.ai.AIMessage'>
[DEBUG] 消息内容: 今天北京天气为多云，气温20°C，适宜外出活动。建议穿着轻便衣物，注意适时增减衣物以防着凉。
[DEBUG] 没有工具调用，直接结束

助手: 今天北京天气为多云，气温20°C，适宜外出活动。建议穿着轻便衣物，注意适时增减衣物以防着凉。


AIMessage(content='今天北京天气为多云，气温20°C，适宜外出活动。建议穿着轻便衣物，注意适时增减衣物以防着凉。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 1940, 'total_tokens': 1972, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen3-235B-A22B-Instruct-2507', 'system_fingerprint': None, 'id': 'chatcmpl-9c08ff6a03bc465ca3a0c9ca7a830287', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--d368b051-53b5-4c23-94de-4826b1d8516f-0', usage_metadata={'input_tokens': 1940, 'output_tokens': 32, 'total_tokens': 1972, 'input_token_details': {}, 'output_token_details': {}})

### 测试 3：数学计算


In [17]:
test_query("帮我算一下 125 * 47 等于多少？")



用户: 帮我算一下 125 * 47 等于多少？
[DEBUG] 检查消息类型: <class 'langchain_core.messages.ai.AIMessage'>
[DEBUG] 消息内容: 
[DEBUG] 发现工具调用: calc_tool
[DEBUG] 检查消息类型: <class 'langchain_core.messages.ai.AIMessage'>
[DEBUG] 消息内容: 125 × 47 = 5875。
[DEBUG] 没有工具调用，直接结束

助手: 125 × 47 = 5875。


AIMessage(content='125 × 47 = 5875。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 3638, 'total_tokens': 3653, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen3-235B-A22B-Instruct-2507', 'system_fingerprint': None, 'id': 'chatcmpl-433fdfde13554a28b9c00f4e38ec6778', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--63b51209-7b9f-4d3b-8296-2cf1c0f37496-0', usage_metadata={'input_tokens': 3638, 'output_tokens': 15, 'total_tokens': 3653, 'input_token_details': {}, 'output_token_details': {}})

### 测试 4：普通对话


In [18]:
test_query("你好，介绍一下你自己吧！")



用户: 你好，介绍一下你自己吧！
[DEBUG] 检查消息类型: <class 'langchain_core.messages.ai.AIMessage'>
[DEBUG] 消息内容: 你好！我是你的智能助手，具备以下能力：

- **天气查询**：可以帮你查询全国各大城市的实时天气情况。
- **数学计算**：支持加、减、乘、除等基本运算，快速帮你算出结果。
- **文档检索**：能从联想官网等信息源中检索你需要的产品或服务信息。

无论你是想了解天气、进行简单计算，还是查找联想相关产品资讯，我都可以为你提供帮助！😊

有什么我可以帮你的吗？
[DEBUG] 没有工具调用，直接结束

助手: 你好！我是你的智能助手，具备以下能力：

- **天气查询**：可以帮你查询全国各大城市的实时天气情况。
- **数学计算**：支持加、减、乘、除等基本运算，快速帮你算出结果。
- **文档检索**：能从联想官网等信息源中检索你需要的产品或服务信息。

无论你是想了解天气、进行简单计算，还是查找联想相关产品资讯，我都可以为你提供帮助！😊

有什么我可以帮你的吗？


AIMessage(content='你好！我是你的智能助手，具备以下能力：\n\n- **天气查询**：可以帮你查询全国各大城市的实时天气情况。\n- **数学计算**：支持加、减、乘、除等基本运算，快速帮你算出结果。\n- **文档检索**：能从联想官网等信息源中检索你需要的产品或服务信息。\n\n无论你是想了解天气、进行简单计算，还是查找联想相关产品资讯，我都可以为你提供帮助！😊\n\n有什么我可以帮你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 6936, 'total_tokens': 7040, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen3-235B-A22B-Instruct-2507', 'system_fingerprint': None, 'id': 'chatcmpl-0dbaffbd827e4111a70df65bf01137c4', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--6911d50a-41a0-45a5-97f1-c41e9f94bcc2-0', usage_metadata={'input_tokens': 6936, 'output_tokens': 104, 'total_tokens': 7040, 'input_token_details': {}, 'output_token_details': {}})

## 9. 交互式运行（可选）

如果你想进行多轮对话，可以使用下面的交互式函数：


In [ ]:
def interactive_chat():
    """交互式对话模式"""
    thread_id = "interactive_1"
    print("开始对话！输入 'q' 或 '退出' 结束对话\n")
    
    while True:
        user_input = input("用户: ")
        if user_input.lower() in ("退出", "q"):
            print("对话结束！")
            break
        
        state = {"messages": [HumanMessage(content=user_input)], "llm_calls": 0}
        config = {"configurable": {"thread_id": thread_id, "user_id": "u123"}}
        
        result = graph.invoke(state, config)["messages"][-1]
        print(f"助手: {result.content}\n")

# 取消注释下面这行来启动交互式对话
# interactive_chat()


## 总结

恭喜！🎉 你已经成功构建了一个功能完整的 LangGraph 应用。

### 关键要点：

1. **工具绑定** - 使用 `@tool` 装饰器定义工具，并用 `.bind_tools()` 绑定到 LLM
2. **状态管理** - 使用 TypedDict 定义状态结构，在节点间传递信息
3. **条件路由** - 根据 LLM 输出动态选择下一个节点
4. **RAG 集成** - 使用向量数据库实现文档检索功能
5. **记忆功能** - 使用 checkpointer 保存对话历史

### 扩展建议：

- 添加更多工具（如网页搜索、数据库查询等）
- 实现更复杂的路由逻辑
- 集成真实的天气 API
- 添加错误处理和重试机制
- 使用流式输出提升用户体验

Happy coding! 🚀
